In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 5 Lab: Counters, Shift Registers & Debouncing

## Course: Accelerated HDL for Digital System Design — Week 2, Session 5

---

## Objectives

By the end of this lab, you will:
- Build a reusable, parameterized debounce module with built-in 2-FF synchronizer
- Construct a shift-register-based LED chase pattern on the Go Board
- Prove that debouncing works by building a reliable button counter
- (Stretch) Build an LFSR pseudo-random pattern generator

## Prerequisites

- Completed Week 1 labs (combinational + sequential fundamentals)
- Watched Day 5 pre-class video (~45 min): counters, shift registers, metastability
- `hex_to_7seg.v` from Day 2 (provided in `starter/`)

## Deliverables

| # | Exercise | Time | What to Submit |
|---|----------|------|----------------|
| 1 | Debounce Module | 30 min | `debounce.v` + `tb_debounce.v` + GTKWave screenshot |
| 2 | LED Chase | 25 min | `led_chase.v` working on Go Board |
| 3 | Button Counter | 25 min | `button_counter.v` — clean count + erratic count comparison |
| 4 | LFSR (stretch) | 20 min | `lfsr_8bit.v` + testbench verifying 255-cycle period |
| 5 | Shift Debounce (stretch) | 20 min | `debounce_shift.v` + comparison analysis |

**Primary deliverable:** Debounced button-controlled LED chase pattern on the Go Board.

---

## Exercise 1: Debounce Module — Build and Simulate (30 min)

**SLOs: 5.3, 5.4**

### Part A: Build `debounce.v`

Create the counter-based debounce module from the pre-class video. Requirements:
- Built-in 2-FF synchronizer (input can be fully asynchronous)
- Parameterized `CLKS_TO_STABLE` (default 250,000 = ~10 ms at 25 MHz)
- When synchronized input differs from clean output, count up
- If count reaches threshold, accept the new value
- If input bounces back before threshold, reset counter

Use the starter file in `starter/debounce.v` — fill in the `YOUR CODE HERE` sections.

### Part B: Simulate with `tb_debounce.v`

The testbench simulates bouncy press/release sequences. Run it:

In [ ]:
!make sim TB=tb_debounce.v SRCS="debounce.v"

### Part C: Verify in GTKWave

1. Count transitions on `clean` — should be exactly 2 (one press, one release)
2. Measure delay from input stabilization to clean output transition
3. Change `CLKS_TO_STABLE` to 5 — does bounce sneak through? Why?

---

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile debounce.v
// =============================================================================
// debounce.v — Counter-Based Button Debouncer (Starter)
// Day 5, Exercise 1
// =============================================================================
// Build a reusable debounce module with built-in 2-FF synchronizer.
// The input i_bouncy can be fully asynchronous.

module debounce #(
    parameter CLKS_TO_STABLE = 250_000  // ~10ms at 25 MHz
)(
    input  wire i_clk,
    input  wire i_bouncy,
    output reg  o_clean
);

    reg [$clog2(CLKS_TO_STABLE)-1:0] r_count;
    reg r_sync_0, r_sync_1;

    always @(posedge i_clk) begin
        // ---- TODO: 2-FF Synchronizer ----
        // Stage 0: sample i_bouncy into r_sync_0
        // Stage 1: sample r_sync_0 into r_sync_1
        // r_sync_1 is now safe to use in synchronous logic


        // ---- TODO: Debounce Counter Logic ----
        // If the synchronized input differs from the current clean output:
        //   - Increment r_count
        //   - If r_count reaches CLKS_TO_STABLE - 1:
        //       Accept the new value (o_clean <= r_sync_1)
        //       Reset counter to 0
        // If the synchronized input matches the clean output:
        //   - Reset counter to 0 (input is stable, no change needed)

    end

endmodule

In [ ]:
%%writefile tb_debounce.v
// =============================================================================
// tb_debounce.v — Testbench for Debounce Module (Starter)
// Day 5, Exercise 1
// =============================================================================

`timescale 1ns / 1ps

module tb_debounce;

    reg  clk;
    reg  bouncy;
    wire clean;

    // Small threshold for simulation
    debounce #(.CLKS_TO_STABLE(10)) uut (
        .i_clk(clk),
        .i_bouncy(bouncy),
        .o_clean(clean)
    );

    // 25 MHz clock (40 ns period)
    initial clk = 0;
    always #20 clk = ~clk;

    initial begin
        $dumpfile("dump.vcd");
        $dumpvars(0, tb_debounce);

        bouncy = 1;  // not pressed (active-low)
        #500;

        // ---- TODO: Simulate a bouncy press ----
        // Toggle bouncy rapidly 5+ times, then settle at 0
        // Example:
        //   bouncy = 0; #60;
        //   bouncy = 1; #40;
        //   bouncy = 0; #80;
        //   ... etc
        //   bouncy = 0;       // final settle
        //   #2000;            // wait beyond threshold


        // ---- TODO: Simulate a bouncy release ----
        // Toggle back to 1 with bouncing, then settle


        // ---- TODO: Verify ----
        // Count transitions on clean — should be exactly 2
        // (one falling edge for press, one rising edge for release)

        $display("=== Debounce simulation complete ===");
        $display("Inspect waveform: clean should have exactly 2 transitions");
        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = debounce
SRCS     = debounce.v
TB       = tb_debounce.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVFLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp

wave: sim
	gtkwave *.vcd &

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_debounce.v debounce.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

## Exercise 2: Shift Register LED Chase (25 min)

**SLOs: 5.2, 5.5**

Build a "Knight Rider" / Cylon LED pattern using a shift register with direction control.

Use `starter/led_chase.v` — the clock divider and debounce instantiations are provided. You implement the bounce-back shift logic.

In [ ]:
!make PROJECT=led_chase TOP=led_chase SRCS="led_chase.v debounce.v"

In [ ]:
!make prog PROJECT=led_chase

**Tasks:**
1. Implement the chase logic (shift with direction reversal at walls)
2. Verify the light sweeps back and forth
3. Verify debounced switch2 controls direction cleanly
4. Experiment with tick rate — make it faster, slower

**Extension:** Make the pattern 2 bits wide (two adjacent LEDs lit).

---

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile debounce.v
// =============================================================================
// debounce.v — Counter-Based Button Debouncer (Starter)
// Day 5, Exercise 1
// =============================================================================
// Build a reusable debounce module with built-in 2-FF synchronizer.
// The input i_bouncy can be fully asynchronous.

module debounce #(
    parameter CLKS_TO_STABLE = 250_000  // ~10ms at 25 MHz
)(
    input  wire i_clk,
    input  wire i_bouncy,
    output reg  o_clean
);

    reg [$clog2(CLKS_TO_STABLE)-1:0] r_count;
    reg r_sync_0, r_sync_1;

    always @(posedge i_clk) begin
        // ---- TODO: 2-FF Synchronizer ----
        // Stage 0: sample i_bouncy into r_sync_0
        // Stage 1: sample r_sync_0 into r_sync_1
        // r_sync_1 is now safe to use in synchronous logic


        // ---- TODO: Debounce Counter Logic ----
        // If the synchronized input differs from the current clean output:
        //   - Increment r_count
        //   - If r_count reaches CLKS_TO_STABLE - 1:
        //       Accept the new value (o_clean <= r_sync_1)
        //       Reset counter to 0
        // If the synchronized input matches the clean output:
        //   - Reset counter to 0 (input is stable, no change needed)

    end

endmodule

In [ ]:
%%writefile led_chase.v
// =============================================================================
// led_chase.v — Shift Register LED Chase Pattern (Starter)
// Day 5, Exercise 2
// =============================================================================

module led_chase (
    input  wire i_clk,
    input  wire i_switch1,   // reset
    input  wire i_switch2,   // direction control
    output wire o_led1,
    output wire o_led2,
    output wire o_led3,
    output wire o_led4
);

    // --- Clock divider for visible speed (~4 Hz) ---
    reg [22:0] r_clk_div;
    wire w_tick = (r_clk_div == 23'd3_124_999);

    always @(posedge i_clk) begin
        if (w_tick)
            r_clk_div <= 0;
        else
            r_clk_div <= r_clk_div + 1;
    end

    // --- Debounce the reset and direction switches ---
    wire w_reset_clean, w_dir_clean;

    debounce #(.CLKS_TO_STABLE(250_000)) db_reset (
        .i_clk(i_clk), .i_bouncy(i_switch1), .o_clean(w_reset_clean)
    );
    debounce #(.CLKS_TO_STABLE(250_000)) db_dir (
        .i_clk(i_clk), .i_bouncy(i_switch2), .o_clean(w_dir_clean)
    );

    wire w_reset = ~w_reset_clean;  // active-low button -> active-high reset

    // --- Shift register with bounce-back ---
    reg [3:0] r_pattern;
    reg       r_dir;  // 0 = shift left, 1 = shift right

    always @(posedge i_clk) begin
        if (w_reset) begin
            r_pattern <= 4'b0001;
            r_dir     <= 1'b0;
        end else if (w_tick) begin
            // ---- TODO: Implement bounce-back chase logic ----
            // Override direction from debounced switch if pressed:
            //   if (!w_dir_clean) r_dir <= ~r_dir;  (toggle on press)
            //
            // Check wall collisions and shift:
            //   If shifting left (r_dir == 0):
            //     If at left wall (r_pattern == 4'b1000), reverse direction
            //     Else shift left: r_pattern <= r_pattern << 1
            //   If shifting right (r_dir == 1):
            //     If at right wall (r_pattern == 4'b0001), reverse direction
            //     Else shift right: r_pattern <= r_pattern >> 1

        end
    end

    // Active-low LED outputs
    assign o_led1 = ~r_pattern[3];
    assign o_led2 = ~r_pattern[2];
    assign o_led3 = ~r_pattern[1];
    assign o_led4 = ~r_pattern[0];

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = led_chase
SRCS     = debounce.v led_chase.v
TB       = 
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

## Exercise 3: Debounced Button Counter (25 min)

**SLOs: 5.4, 5.5**

The definitive test: prove debouncing works by building a press counter.

Use `starter/button_counter.v` — provided complete (this is the reference integration).

### The Test Protocol

1. **With debounce:** Press button 16 times slowly. Display should count 0→1→2→...→F→0 cleanly.
2. **Without debounce:** Modify to bypass debounce (use direct sync only). Repeat. Record the erratic count.
3. **Record:** "With debounce: 16 presses → count reached 0 (wrapped). Without: 16 presses → count reached [X]."

In [ ]:
!make PROJECT=button_counter TOP=button_counter SRCS="button_counter.v debounce.v hex_to_7seg.v"

In [ ]:
!make prog PROJECT=button_counter

---

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile button_counter.v
// =============================================================================
// button_counter.v — Debounced Button Counter (Provided)
// Day 5, Exercise 3
// =============================================================================
// This module is provided complete. Your task:
//   1. Build and program it — verify clean counting (0..F)
//   2. Modify to bypass debounce — observe erratic counting
//   3. Record your comparison results

module button_counter (
    input  wire i_clk,
    input  wire i_switch1,   // reset
    input  wire i_switch2,   // count button
    output wire o_segment1_a, o_segment1_b, o_segment1_c,
    output wire o_segment1_d, o_segment1_e, o_segment1_f,
    output wire o_segment1_g,
    output wire o_led1       // heartbeat
);

    // --- Debounce both buttons ---
    wire w_reset_clean, w_count_clean;

    debounce #(.CLKS_TO_STABLE(250_000)) db_reset (
        .i_clk(i_clk), .i_bouncy(i_switch1), .o_clean(w_reset_clean)
    );
    debounce #(.CLKS_TO_STABLE(250_000)) db_count (
        .i_clk(i_clk), .i_bouncy(i_switch2), .o_clean(w_count_clean)
    );

    // --- Edge detector: press = falling edge of active-low ---
    reg r_count_prev;
    always @(posedge i_clk)
        r_count_prev <= w_count_clean;

    wire w_count_press = ~w_count_clean & r_count_prev;

    // --- 4-bit counter ---
    wire w_reset = ~w_reset_clean;
    reg [3:0] r_count;

    always @(posedge i_clk) begin
        if (w_reset)
            r_count <= 4'd0;
        else if (w_count_press)
            r_count <= r_count + 4'd1;
    end

    // --- 7-segment display ---
    wire [6:0] w_seg;
    hex_to_7seg decoder (.i_hex(r_count), .o_seg(w_seg));

    assign {o_segment1_a, o_segment1_b, o_segment1_c,
            o_segment1_d, o_segment1_e, o_segment1_f,
            o_segment1_g} = w_seg;

    // --- Heartbeat ---
    reg [23:0] r_hb_count;
    always @(posedge i_clk)
        r_hb_count <= r_hb_count + 1;
    assign o_led1 = ~r_hb_count[23];

endmodule

In [ ]:
%%writefile debounce.v
// =============================================================================
// debounce.v — Counter-Based Button Debouncer (Starter)
// Day 5, Exercise 1
// =============================================================================
// Build a reusable debounce module with built-in 2-FF synchronizer.
// The input i_bouncy can be fully asynchronous.

module debounce #(
    parameter CLKS_TO_STABLE = 250_000  // ~10ms at 25 MHz
)(
    input  wire i_clk,
    input  wire i_bouncy,
    output reg  o_clean
);

    reg [$clog2(CLKS_TO_STABLE)-1:0] r_count;
    reg r_sync_0, r_sync_1;

    always @(posedge i_clk) begin
        // ---- TODO: 2-FF Synchronizer ----
        // Stage 0: sample i_bouncy into r_sync_0
        // Stage 1: sample r_sync_0 into r_sync_1
        // r_sync_1 is now safe to use in synchronous logic


        // ---- TODO: Debounce Counter Logic ----
        // If the synchronized input differs from the current clean output:
        //   - Increment r_count
        //   - If r_count reaches CLKS_TO_STABLE - 1:
        //       Accept the new value (o_clean <= r_sync_1)
        //       Reset counter to 0
        // If the synchronized input matches the clean output:
        //   - Reset counter to 0 (input is stable, no change needed)

    end

endmodule

In [ ]:
%%writefile hex_to_7seg.v
// hex_to_7seg.v — Provided module (from Day 2)
module hex_to_7seg (
    input  wire [3:0] i_hex,
    output reg  [6:0] o_seg   // {a,b,c,d,e,f,g} active low
);
    always @(*) begin
        case (i_hex)
            4'h0: o_seg = 7'b0000001;  4'h1: o_seg = 7'b1001111;
            4'h2: o_seg = 7'b0010010;  4'h3: o_seg = 7'b0000110;
            4'h4: o_seg = 7'b1001100;  4'h5: o_seg = 7'b0100100;
            4'h6: o_seg = 7'b0100000;  4'h7: o_seg = 7'b0001111;
            4'h8: o_seg = 7'b0000000;  4'h9: o_seg = 7'b0000100;
            4'hA: o_seg = 7'b0001000;  4'hB: o_seg = 7'b1100000;
            4'hC: o_seg = 7'b0110001;  4'hD: o_seg = 7'b1000010;
            4'hE: o_seg = 7'b0110000;  4'hF: o_seg = 7'b0111000;
        endcase
    end
endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = button_counter
SRCS     = button_counter.v debounce.v hex_to_7seg.v
TB       = 
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

## Exercise 4 (Stretch): LFSR Pattern Generator (20 min)

**SLO: 5.6**

Build `lfsr_8bit.v` — an 8-bit Linear Feedback Shift Register with maximal-length taps.

Use `starter/lfsr_8bit.v`. Create a top module that:
- Clocks the LFSR at ~4 Hz using a tick
- Displays lower 4 bits on 7-seg, upper 4 on LEDs
- Uses a button to pause/resume

**Simulation:** Write a testbench verifying 255-cycle period (returns to seed after exactly 255 enables).

---

#### 📝 Exercise 4 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile hex_to_7seg.v
// hex_to_7seg.v — Provided module (from Day 2)
module hex_to_7seg (
    input  wire [3:0] i_hex,
    output reg  [6:0] o_seg   // {a,b,c,d,e,f,g} active low
);
    always @(*) begin
        case (i_hex)
            4'h0: o_seg = 7'b0000001;  4'h1: o_seg = 7'b1001111;
            4'h2: o_seg = 7'b0010010;  4'h3: o_seg = 7'b0000110;
            4'h4: o_seg = 7'b1001100;  4'h5: o_seg = 7'b0100100;
            4'h6: o_seg = 7'b0100000;  4'h7: o_seg = 7'b0001111;
            4'h8: o_seg = 7'b0000000;  4'h9: o_seg = 7'b0000100;
            4'hA: o_seg = 7'b0001000;  4'hB: o_seg = 7'b1100000;
            4'hC: o_seg = 7'b0110001;  4'hD: o_seg = 7'b1000010;
            4'hE: o_seg = 7'b0110000;  4'hF: o_seg = 7'b0111000;
        endcase
    end
endmodule

In [ ]:
%%writefile lfsr_8bit.v
// =============================================================================
// lfsr_8bit.v — 8-bit LFSR Pseudo-Random Generator (Starter)
// Day 5, Exercise 4 (Stretch)
// =============================================================================

module lfsr_8bit (
    input  wire       i_clk,
    input  wire       i_reset,
    input  wire       i_enable,
    output reg  [7:0] o_lfsr,
    output wire       o_valid  // 0 when stuck in all-zeros
);

    // Taps for maximal-length 8-bit LFSR: x^8 + x^6 + x^5 + x^4 + 1
    wire w_feedback = o_lfsr[7] ^ o_lfsr[5] ^ o_lfsr[4] ^ o_lfsr[3];

    always @(posedge i_clk) begin
        if (i_reset)
            o_lfsr <= 8'h01;  // nonzero seed
        else if (i_enable)
            o_lfsr <= {o_lfsr[6:0], w_feedback};
    end

    assign o_valid = |o_lfsr;

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = lfsr_8bit
SRCS     = hex_to_7seg.v lfsr_8bit.v
TB       = 
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

## Exercise 5 (Stretch): Shift Register Debounce (20 min)

**SLOs: 5.2, 5.4**

Implement `debounce_shift.v` — an alternative architecture that samples the input into an N-bit shift register at a slow rate. If all N bits agree, the input is stable.

**Comparison task:** Which approach (counter vs. shift register) uses more resources? Which responds faster? Which is easier to tune?

---

## Build Commands Quick Reference

In [ ]:
!make sim TB=tb_debounce.v SRCS="debounce.v"         # Simulate

In [ ]:
# Render waveforms inline (replaces GTKWave)
show_waves('dump.vcd')

In [ ]:
!make PROJECT=led_chase SRCS="led_chase.v debounce.v"  # Synthesize

In [ ]:
!make prog PROJECT=led_chase                            # Program board

In [ ]:
!make stat PROJECT=led_chase SRCS="led_chase.v debounce.v"  # Resources